In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.decomposition import PCA

from Src.data import load_csv_df, save_csv_df
from Src.plotting import set_plotting_style

In [ ]:
# Already prepared accent palette applying using function from mine Src module (plotting)
set_plotting_style()

In [ ]:
DF_PATH = "../Data/Raw/raw-data.csv"

# Safely loading dataframe using function from mine Src module (data)
df = load_csv_df(
    df_path=DF_PATH,
)

In [ ]:
# Dropping CUST_ID because it has a lot of unique values (See in EDA)
df = df.drop(columns=["CUST_ID"])

In [ ]:
# Filling 314 NaN Total with medians (Read more in EDA)
df["CREDIT_LIMIT"] = df["CREDIT_LIMIT"].fillna(df["CREDIT_LIMIT"].median())
df["MINIMUM_PAYMENTS"] = df["MINIMUM_PAYMENTS"].fillna(df["MINIMUM_PAYMENTS"].median())

In [ ]:
NUMERIC_COL = df.select_dtypes(exclude=["object", "string", "category"]).columns

# Why these columns (See in EDA hist plots analysis)
EXCLUDE_COLS = [
    "TENURE",
    "PURCHASES_FREQUENCY",
    "BALANCE_FREQUENCY",
    "ONEOFF_PURCHASES_FREQUENCY",
    "CASH_ADVANCE_FREQUENCY",
    "PURCHASES_INSTALLMENTS_FREQUENCY",
    "PRC_FULL_PAYMENT",
]
TARGET_COLS = NUMERIC_COL.difference(EXCLUDE_COLS, sort=False).tolist()
print(TARGET_COLS)

In [ ]:
# Log scaling to remove all skews and outliers
preprocessor = ColumnTransformer(
    transformers=[
        ("log1p", FunctionTransformer(np.log1p), TARGET_COLS),
        ("passthrough", "passthrough", EXCLUDE_COLS)
    ]
)
x_log = preprocessor.fit_transform(df)

# Scaling data for correct clustering
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_log)

In [ ]:
pca_full = PCA(random_state=42)
pca_full = pca_full.fit(x_scaled)
cum_varies = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(10, 8))
sns.lineplot(
    x=range(1, len(cum_varies) + 1),
    y=cum_varies,
    marker="o",
    linewidth=2,
)

plt.axhline(y=0.8, color="red", linestyle="-", label="80% Variance cutoff")
plt.title("PCA Cumulative Explained Variance Ration")
plt.xlabel("Number of PCA Components")
plt.ylabel("Cumulative Explained Variance Ratio")
plt.xticks(range(1, len(cum_varies) + 1))
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

var_6 = pca_full.explained_variance_ratio_[5] * 100
cum_var_6 = sum(pca_full.explained_variance_ratio_[:6]) * 100

print(f"n_components 6 variance ration: {var_6:.2f}%")
print(f"Cumulative varince (1-6): {cum_var_6:.2f}%")

# The best n_components is 6 (Variance ration summa = 84.17%)
BEST_N_COMPONENTS = 6

In [ ]:
pca = PCA(
    n_components=BEST_N_COMPONENTS,
    random_state=42
)
x_pca = pca.fit_transform(x_scaled)

df_pca = pd.DataFrame(
    x_pca,
    columns=[f"PCA{i+1}" for i in range(BEST_N_COMPONENTS)]
)

display(df_pca.head(3))
display(df_pca.tail(3))

In [ ]:
OUTPUT_PATH = "../Data/Processed/processed-data.csv"

# Safely saving dataframe using function from mine Src module (data)
save_csv_df(
    df=df_pca,
    output_path=OUTPUT_PATH,
)